In [ ]:
# ============================================================
# CELDA 1: Carga del Dataset de Imágenes Histopatológicas
# ============================================================
# Se descarga el dataset desde Kaggle con kagglehub.
# Cada imagen se redimensiona a 16x16 píxeles en RGB y se almacena
# en X_raw (imágenes) e y_raw (etiquetas numéricas 0,1,2).
# Las etiquetas se usan en supervisado para entrenar, y en no
# supervisado solo para evaluar la calidad del clustering.

import os
import kagglehub
import cv2
import numpy as np

dataset_path = kagglehub.dataset_download("rm1000/lung-cancer-histopathological-images")
clases = sorted(os.listdir(dataset_path))
X_raw, y_raw = [], []

for label, clase in enumerate(clases):
    ruta = os.path.join(dataset_path, clase)
    for file in os.listdir(ruta):
        img_path = os.path.join(ruta, file)
        img = cv2.imread(img_path)
        img = cv2.resize(img, (16, 16))
        X_raw.append(img)
        y_raw.append(label)

X_raw  = np.array(X_raw)
y_raw  = np.array(y_raw)
X_flat = X_raw.reshape(len(X_raw), -1)   # shape: [15000, 768]

print(f"Imágenes cargadas   : {len(X_raw)}")
print(f"Shape aplanado      : {X_flat.shape}")
print(f"Valores nulos       : {np.isnan(X_flat.astype(float)).sum()}")
print(f"Distribución clases : {np.bincount(y_raw)}")


In [ ]:
# ============================================================
# CELDA 2: Train/Test Split (80/20) y Normalización StandardScaler
# ============================================================
# Se divide el dataset en 80% train y 20% test manteniendo proporción
# de clases (stratify). La normalización se hace con StandardScaler:
# fit_transform SOLO en train para evitar data leakage,
# y transform (sin fit) en test.

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_flat, y_raw,
    test_size=0.2,
    stratify=y_raw,
    random_state=42
)

# Normalización Z-score con StandardScaler (fit SOLO en train)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

print(f"Train shape : {X_train_scaled.shape}  |  Test shape : {X_test_scaled.shape}")
print(f"Clases train: {np.bincount(y_train)}")
print(f"Clases test : {np.bincount(y_test)}")


In [ ]:
# ============================================================
# CELDA 3: PCA(100) para entrenar + t-SNE solo para visualizar
# ============================================================
# PASO 1 – PCA(100): fit sobre train, transform en train y test.
#   X_train_pca y X_test_pca son los datos reales con los que
#   se entrenarán y evaluarán TODOS los modelos supervisados
#   y no supervisados.
#
# PASO 2 – t-SNE: SOLO para visualización. Se aplica sobre una
#   muestra de X_train_pca para ver en 2D la separabilidad
#   de las clases. NO se usa para entrenar ningún modelo.

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# --- PASO 1: PCA(100) → estos son los nuevos X_train y X_test ---
pca = PCA(n_components=100, random_state=42)
pca.fit(X_train_scaled)                        # fit SOLO en train

X_train_pca = pca.transform(X_train_scaled)    # [12000, 100] ← para entrenar
X_test_pca  = pca.transform(X_test_scaled)     # [3000,  100] ← para evaluar

varianza_acum = pca.explained_variance_ratio_.cumsum()[-1]
print(f"PCA(100) – Varianza explicada acumulada: {varianza_acum:.2%}")
print(f"Train PCA shape : {X_train_pca.shape}")
print(f"Test  PCA shape : {X_test_pca.shape}")

# --- PASO 2: t-SNE sobre muestra de X_train_pca (SOLO visualización) ---
np.random.seed(42)
idx_vis = np.random.choice(len(X_train_pca), size=3000, replace=False)
X_vis   = X_train_pca[idx_vis]
y_vis   = y_train[idx_vis]

tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, verbose=1)
X_tsne = tsne.fit_transform(X_vis)

colores = ['#e74c3c', '#3498db', '#2ecc71']
plt.figure(figsize=(10, 7))
for i in range(3):
    mask = y_vis == i
    plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=colores[i], label=f'Clase {i}', alpha=0.5, s=10)
plt.title('t-SNE sobre PCA(100) – Solo visualización (muestra 3000)')
plt.xlabel('t-SNE Dim 1'); plt.ylabel('t-SNE Dim 2')
plt.legend(markerscale=3); plt.grid(True, alpha=0.2)
plt.show()


---
## PARTE 1: APRENDIZAJE SUPERVISADO
### Entrada para todos los modelos: X_train_pca y X_test_pca (PCA 100 componentes)
---

In [ ]:
# ============================================================
# CELDA 4: Optimización Bayesiana – Decision Tree, Random Forest y SVC
# ============================================================
# Se usa BayesSearchCV (skopt) con los MISMOS espacios de búsqueda
# y parámetros del notebook de referencia (adelanto-segunda-parte)
# para poder comparar resultados directamente.
#
# Espacios idénticos al notebook original:
#   space_dt  → criterion, max_depth, min_samples_leaf, min_samples_split, class_weight
#   space_rf  → n_estimators, max_depth, min_samples_leaf, max_features, class_weight
#   space_svc → C (log-uniform), gamma, kernel, class_weight
#
# BayesSearchCV: n_iter=10, cv=2, n_jobs=-1 (igual al notebook original)
# Los datos de entrada son X_train_pca (100 componentes PCA) en lugar
# de X_train_rfe (50 features por importancia) del notebook original.

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer

# ==========================================================
# FUNCIÓN DE OPTIMIZACIÓN BAYESIANA (igual al notebook original)
# ==========================================================
def ejecutar_bayes_enfocado(nombre, estimator, search_space, X_tr, y_tr):
    print(f"--- Optimizando {nombre} ---")
    opt = BayesSearchCV(
        estimator=estimator,
        search_spaces=search_space,
        n_iter=10,
        cv=2,
        n_jobs=-1,
        random_state=42,
        return_train_score=True
    )
    opt.fit(X_tr.astype('float32'), y_tr)
    return opt

# ==========================================================
# ESPACIOS DE BÚSQUEDA (exactamente iguales al notebook original)
# ==========================================================
space_dt = {
    'criterion'        : Categorical(['gini', 'entropy']),
    'max_depth'        : Integer(10, 20),
    'min_samples_leaf' : Integer(20, 50),
    'min_samples_split': Integer(50, 100),
    'class_weight'     : Categorical(['balanced', None])
}

space_rf = {
    'n_estimators'     : Integer(50, 150),
    'max_depth'        : Integer(10, 20),
    'min_samples_leaf' : Integer(30, 50),
    'max_features'     : Categorical(['sqrt', 'log2']),
    'class_weight'     : Categorical(['balanced', None])
}

space_svc = {
    'C'           : Real(0.1, 10, prior='log-uniform'),
    'gamma'       : Categorical(['scale', 'auto', 0.01, 0.001]),
    'kernel'      : Categorical(['rbf', 'poly', 'sigmoid']),
    'class_weight': Categorical(['balanced', None])
}

# ==========================================================
# EJECUCIÓN (entrada: X_train_pca en lugar de X_train_rfe)
# ==========================================================
opt_dt  = ejecutar_bayes_enfocado("Decision Tree", DecisionTreeClassifier(random_state=42),                        space_dt,  X_train_pca, y_train)
opt_rf  = ejecutar_bayes_enfocado("Random Forest", RandomForestClassifier(random_state=42),                       space_rf,  X_train_pca, y_train)
opt_svc = ejecutar_bayes_enfocado("SVC",           SVC(random_state=42, cache_size=1000, max_iter=10000, tol=1e-3), space_svc, X_train_pca, y_train)

print("\nMejores parámetros encontrados:")
print(f"  DT  → {opt_dt.best_params_}")
print(f"  RF  → {opt_rf.best_params_}")
print(f"  SVC → {opt_svc.best_params_}")


In [ ]:
# ============================================================
# CELDA 5: Evaluación Decision Tree – Matriz, Reporte y Brecha
# ============================================================
# Se evalúa el mejor estimador encontrado por BayesSearchCV para
# el Decision Tree sobre X_test_pca.
# Métricas idénticas al notebook original:
#   - Matriz de confusión (colormap Blues)
#   - Classification report (precision, recall, f1, accuracy)
#   - Brecha train/validación (mean_train_score vs mean_test_score)

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

best_dt   = opt_dt.best_estimator_
y_pred_dt = best_dt.predict(X_test_pca)

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_dt), annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión – Decision Tree (Bayesiano)')
plt.ylabel('Realidad'); plt.xlabel('Predicción')
plt.show()

print("--- REPORTE DE CLASIFICACIÓN – Decision Tree ---")
print(classification_report(y_test, y_pred_dt))

res_dt   = pd.DataFrame(opt_dt.cv_results_)
best_row = res_dt.loc[opt_dt.best_index_]
train_acc = best_row['mean_train_score']
val_acc   = best_row['mean_test_score']
print(f"Accuracy Entrenamiento (Promedio): {train_acc:.4f}")
print(f"Accuracy Validación   (Promedio): {val_acc:.4f}")
print(f"Diferencia (Brecha)             : {train_acc - val_acc:.4f}")


In [ ]:
# ============================================================
# CELDA 6: Evaluación Random Forest – Matriz, Reporte y Brecha
# ============================================================
# Se evalúa el mejor estimador de Random Forest (BayesSearchCV)
# sobre X_test_pca.
# Mismas métricas que el notebook original:
#   - Matriz de confusión (colormap Greens)
#   - Classification report
#   - Brecha train/validación

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

best_rf   = opt_rf.best_estimator_
y_pred_rf = best_rf.predict(X_test_pca)

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens')
plt.title('Matriz de Confusión – Random Forest (Bayesiano)')
plt.ylabel('Realidad'); plt.xlabel('Predicción')
plt.show()

print("--- REPORTE DE CLASIFICACIÓN – Random Forest ---")
print(classification_report(y_test, y_pred_rf))

res_rf   = pd.DataFrame(opt_rf.cv_results_)
best_row = res_rf.loc[opt_rf.best_index_]
train_acc = best_row['mean_train_score']
val_acc   = best_row['mean_test_score']
print(f"Accuracy Entrenamiento (Promedio): {train_acc:.4f}")
print(f"Accuracy Validación   (Promedio): {val_acc:.4f}")
print(f"Diferencia (Brecha)             : {train_acc - val_acc:.4f}")


In [ ]:
# ============================================================
# CELDA 7: Evaluación SVC – Matriz, Reporte y Brecha
# ============================================================
# Se evalúa el mejor estimador de SVC (BayesSearchCV)
# sobre X_test_pca.
# Mismas métricas que el notebook original:
#   - Matriz de confusión (colormap Purples)
#   - Classification report
#   - Brecha train/validación

from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

best_svc   = opt_svc.best_estimator_
y_pred_svc = best_svc.predict(X_test_pca)

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_svc), annot=True, fmt='d', cmap='Purples')
plt.title('Matriz de Confusión – SVC (Bayesiano)')
plt.ylabel('Realidad'); plt.xlabel('Predicción')
plt.show()

print("--- REPORTE DE CLASIFICACIÓN – SVC ---")
print(classification_report(y_test, y_pred_svc))

res_svc  = pd.DataFrame(opt_svc.cv_results_)
best_row = res_svc.loc[opt_svc.best_index_]
train_acc = best_row['mean_train_score']
val_acc   = best_row['mean_test_score']
print(f"Accuracy Entrenamiento (Promedio): {train_acc:.4f}")
print(f"Accuracy Validación   (Promedio): {val_acc:.4f}")
print(f"Diferencia (Brecha)             : {train_acc - val_acc:.4f}")


In [ ]:
# ============================================================
# CELDA 8: Red Neuronal Profunda (DNN) con Keras/TensorFlow
# ============================================================
# DNN secuencial sobre X_train_pca (100 componentes PCA).
# Arquitectura idéntica al notebook original:
#   Input(100) → Dense(128, relu) → Dense(64, relu) → Dense(32, relu) → Softmax(3)
# - Compilación: Adam + categorical_crossentropy
# - EarlyStopping: patience=5, restaura mejores pesos
# - Entrenamiento: máx 50 épocas, batch=32, 20% validación interna
# Métricas: curvas loss/accuracy, matriz de confusión (Reds),
# classification report y curvas ROC por clase.

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix, classification_report, RocCurveDisplay
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

X_tr = X_train_pca.astype('float32')
X_te = X_test_pca.astype('float32')
y_tr = keras.utils.to_categorical(y_train, num_classes=3)
y_te = keras.utils.to_categorical(y_test,  num_classes=3)

# Arquitectura (igual al notebook original)
model_dnn = keras.Sequential([
    keras.Input(shape=(X_tr.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dense(64,  activation='relu'),
    layers.Dense(32,  activation='relu'),
    layers.Dense(3,   activation='softmax')
])
model_dnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_dnn.summary()

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model_dnn.fit(
    X_tr, y_tr,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

# Curvas de entrenamiento
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('DNN – Accuracy por época')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('DNN – Loss por época')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Matriz de confusión y reporte
y_pred_prob = model_dnn.predict(X_te)
y_pred_dnn  = np.argmax(y_pred_prob, axis=1)
y_true_dnn  = np.argmax(y_te, axis=1)

plt.figure(figsize=(5, 4))
sns.heatmap(confusion_matrix(y_true_dnn, y_pred_dnn), annot=True, fmt='d', cmap='Reds', cbar=False)
plt.title('Matriz de Confusión – DNN'); plt.ylabel('Real'); plt.xlabel('Predicción')
plt.show()

print("--- REPORTE DE CLASIFICACIÓN – DNN ---")
print(classification_report(y_true_dnn, y_pred_dnn))

# Curvas ROC
y_test_bin = label_binarize(y_true_dnn, classes=[0, 1, 2])
fig, ax = plt.subplots(figsize=(7, 5))
for i in range(3):
    RocCurveDisplay.from_predictions(y_test_bin[:, i], y_pred_prob[:, i], name=f"Clase {i}", ax=ax)
ax.set_title('Curvas ROC – DNN'); plt.show()


In [ ]:
# ============================================================
# CELDA 9: Comparativa Final Supervisado – K-Fold y Test Accuracy
# ============================================================
# Estructura idéntica a la celda 18 del notebook original.
# SECCIÓN K-FOLD: StratifiedKFold(3) sobre X_train_pca para
#   DT, RF y SVC. La DNN se evalúa fold a fold con model.evaluate().
#   Resultado: accuracy promedio ± std ordenado de mayor a menor.
# SECCIÓN TEST FINAL: accuracy directo sobre X_test_pca para
#   cada modelo, ordenado de mayor a menor.

from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# --- K-Fold ML ---
print("=== COMPARATIVA FINAL – K-Fold (3 folds) ===")
comparativa_ml = [
    ('Decision Tree', opt_dt.best_estimator_),
    ('Random Forest', opt_rf.best_estimator_),
    ('SVC',          opt_svc.best_estimator_),
]

resultados_kf = []
for nombre, modelo in comparativa_ml:
    scores = cross_val_score(modelo, X_train_pca.astype('float32'), y_train, cv=kf, scoring='accuracy')
    resultados_kf.append((nombre, scores.mean(), scores.std()))

# --- K-Fold DNN ---
dnn_scores = []
for train_idx, val_idx in kf.split(X_tr, y_train):
    y_val_cat = keras.utils.to_categorical(y_train[val_idx], 3)
    _, acc = model_dnn.evaluate(X_tr[val_idx], y_val_cat, verbose=0)
    dnn_scores.append(acc)
dnn_arr = np.array(dnn_scores)
resultados_kf.append(('DNN', dnn_arr.mean(), dnn_arr.std()))

for nombre, mean, std in sorted(resultados_kf, key=lambda x: x[1], reverse=True):
    print(f"{nombre:20s} → {mean:.4f} ± {std:.4f}")

# --- Test Accuracy Final ---
print("\n=== TEST ACCURACY FINAL ===")
comparativa_test = [
    ('Decision Tree', opt_dt.best_estimator_.score(X_test_pca, y_test)),
    ('Random Forest', opt_rf.best_estimator_.score(X_test_pca, y_test)),
    ('SVC',          opt_svc.best_estimator_.score(X_test_pca, y_test)),
    ('DNN',          model_dnn.evaluate(X_te, y_te, verbose=0)[1]),
]
for nombre, acc in sorted(comparativa_test, key=lambda x: x[1], reverse=True):
    print(f"{nombre:20s} → {acc:.4f}")


---
## PARTE 2: APRENDIZAJE NO SUPERVISADO
### Ronda A – Datos crudos normalizados (X_train_scaled, 768D)
### Ronda B – Datos reducidos con PCA(100) (X_train_pca, 100D)
---

In [ ]:
# ============================================================
# CELDA 10 – RONDA A: K-Means sobre datos crudos normalizados
# ============================================================
# K-Means con K=3 sobre X_train_scaled (768 features).
# Primero el Método del Codo (sobre muestra) para confirmar K.
# Métricas: Silhouette Score (calidad interna sin etiquetas) y
# ARI (comparación con las clases reales).
# En alta dimensión el Silhouette es bajo por la maldición de la
# dimensionalidad; la Ronda B (PCA) mostrará la mejora.

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Método del Codo (muestra de 3000 para velocidad)
np.random.seed(42)
idx_m = np.random.choice(len(X_train_scaled), size=3000, replace=False)
X_m   = X_train_scaled[idx_m]

inercias, rango_k = [], range(2, 9)
for k in rango_k:
    km_tmp = KMeans(n_clusters=k, random_state=42, n_init=5)
    km_tmp.fit(X_m)
    inercias.append(km_tmp.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(rango_k, inercias, marker='o', color='steelblue')
plt.axvline(x=3, color='red', linestyle='--', alpha=0.5, label='K=3')
plt.title('[RONDA A] Método del Codo – K-Means (datos crudos 768D)')
plt.xlabel('K'); plt.ylabel('Inercia'); plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

# K-Means K=3 sobre todo el train
km_a = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km_a = km_a.fit_predict(X_train_scaled)

sil_km_a = silhouette_score(X_train_scaled, labels_km_a, sample_size=3000, random_state=42)
ari_km_a = adjusted_rand_score(y_train, labels_km_a)

print("=== [RONDA A] K-Means (768D) ===")
print(f"Silhouette Score : {sil_km_a:.4f}")
print(f"ARI              : {ari_km_a:.4f}")
print(f"Distribución     : {np.bincount(labels_km_a)}")

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_train, labels_km_a), annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'C{i}' for i in range(3)],
            yticklabels=[f'Clase {i}' for i in range(3)])
plt.title('[RONDA A] K-Means: Clusters vs Clases Reales')
plt.ylabel('Clase Real'); plt.xlabel('Cluster')
plt.show()


In [ ]:
# ============================================================
# CELDA 11 – RONDA A: DBSCAN sobre datos crudos normalizados
# ============================================================
# DBSCAN sobre X_train_scaled (768D).
# Gráfico K-distancias (k=10) sobre muestra para elegir eps.
# En 768D las distancias son grandes; eps_a parte de 15.0.
# Métricas: clusters encontrados, % ruido, Silhouette (no-ruido), ARI.

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)
idx_db = np.random.choice(len(X_train_scaled), size=2000, replace=False)
X_db   = X_train_scaled[idx_db]

min_s = 10
nn = NearestNeighbors(n_neighbors=min_s)
nn.fit(X_db)
dists, _ = nn.kneighbors(X_db)
dists_sorted = np.sort(dists[:, -1])[::-1]

plt.figure(figsize=(9, 4))
plt.plot(dists_sorted, color='darkorange')
plt.title(f'[RONDA A] K-distancias (k={min_s}) – datos crudos 768D')
plt.xlabel('Puntos ordenados'); plt.ylabel('Distancia')
plt.grid(True, alpha=0.3); plt.show()

eps_a = 15.0   # <-- Modificar según el codo del gráfico

dbscan_a    = DBSCAN(eps=eps_a, min_samples=min_s, n_jobs=-1)
labels_db_a = dbscan_a.fit_predict(X_train_scaled)

n_clust_a = len(set(labels_db_a)) - (1 if -1 in labels_db_a else 0)
n_ruido_a = np.sum(labels_db_a == -1)

print("=== [RONDA A] DBSCAN (768D) ===")
print(f"eps={eps_a}, min_samples={min_s}")
print(f"Clusters encontrados : {n_clust_a}")
print(f"Puntos de ruido      : {n_ruido_a} ({n_ruido_a/len(labels_db_a)*100:.1f}%)")

mask_nr_a = labels_db_a != -1
if n_clust_a >= 2 and mask_nr_a.sum() > 1:
    sil_db_a = silhouette_score(X_train_scaled[mask_nr_a], labels_db_a[mask_nr_a],
                                 sample_size=3000, random_state=42)
    print(f"Silhouette Score     : {sil_db_a:.4f}")
else:
    sil_db_a = float('nan')
    print("Silhouette Score     : N/A (ajusta eps)")

ari_db_a = adjusted_rand_score(y_train, labels_db_a)
print(f"ARI                  : {ari_db_a:.4f}")


In [ ]:
# ============================================================
# CELDA 12 – RONDA A: Clustering Jerárquico sobre datos crudos
# ============================================================
# AgglomerativeClustering con n_clusters=3, linkage='ward'
# sobre X_train_scaled (768D).
# Ward minimiza la varianza intra-cluster al fusionar grupos.
# Métricas: Silhouette Score, ARI y matriz de confusión (verde).

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

agg_a        = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg_a = agg_a.fit_predict(X_train_scaled)

sil_agg_a = silhouette_score(X_train_scaled, labels_agg_a, sample_size=3000, random_state=42)
ari_agg_a = adjusted_rand_score(y_train, labels_agg_a)

print("=== [RONDA A] Clustering Jerárquico Ward (768D) ===")
print(f"Silhouette Score : {sil_agg_a:.4f}")
print(f"ARI              : {ari_agg_a:.4f}")
print(f"Distribución     : {np.bincount(labels_agg_a)}")

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_train, labels_agg_a), annot=True, fmt='d', cmap='Greens',
            xticklabels=[f'C{i}' for i in range(3)],
            yticklabels=[f'Clase {i}' for i in range(3)])
plt.title('[RONDA A] Jerárquico Ward: Clusters vs Clases Reales')
plt.ylabel('Clase Real'); plt.xlabel('Cluster')
plt.show()


In [ ]:
# ============================================================
# CELDA 13 – RONDA B: K-Means sobre PCA(100)
# ============================================================
# K-Means con K=3 sobre X_train_pca (100 componentes).
# Se espera mejor Silhouette que Ronda A al trabajar en espacio
# de menor dimensión con mayor varianza concentrada.
# Visualización 2D con las 2 primeras componentes PCA.
# Métricas: Silhouette Score, ARI y matriz de confusión.

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

km_b        = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_km_b = km_b.fit_predict(X_train_pca)

sil_km_b = silhouette_score(X_train_pca, labels_km_b, sample_size=3000, random_state=42)
ari_km_b = adjusted_rand_score(y_train, labels_km_b)

print("=== [RONDA B] K-Means (PCA 100) ===")
print(f"Silhouette Score : {sil_km_b:.4f}")
print(f"ARI              : {ari_km_b:.4f}")
print(f"Distribución     : {np.bincount(labels_km_b)}")

# Visualización 2D (PC1 vs PC2)
colores = ['#e74c3c', '#3498db', '#2ecc71']
centroides_2d = km_b.cluster_centers_[:, :2]

plt.figure(figsize=(9, 6))
for c in range(3):
    mask = labels_km_b == c
    plt.scatter(X_train_pca[mask, 0], X_train_pca[mask, 1],
                c=colores[c], label=f'Cluster {c}', alpha=0.3, s=8)
plt.scatter(centroides_2d[:, 0], centroides_2d[:, 1],
            c='black', marker='X', s=200, label='Centroides', zorder=5)
plt.title('[RONDA B] K-Means: Clusters en PCA 2D')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(markerscale=2); plt.grid(True, alpha=0.2)
plt.show()

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_train, labels_km_b), annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'C{i}' for i in range(3)],
            yticklabels=[f'Clase {i}' for i in range(3)])
plt.title('[RONDA B] K-Means: Clusters vs Clases Reales')
plt.ylabel('Clase Real'); plt.xlabel('Cluster')
plt.show()


In [ ]:
# ============================================================
# CELDA 14 – RONDA B: DBSCAN sobre PCA(100)
# ============================================================
# DBSCAN sobre X_train_pca (100D), donde las distancias son
# mucho más manejables que en 768D.
# Gráfico K-distancias para elegir eps en el nuevo espacio.
# eps_b parte de 3.5 (espacio PCA normalizado).
# Métricas: clusters encontrados, % ruido, Silhouette, ARI.

from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, adjusted_rand_score
import matplotlib.pyplot as plt
import numpy as np

np.random.seed(42)
idx_db2 = np.random.choice(len(X_train_pca), size=2000, replace=False)
X_db2   = X_train_pca[idx_db2]

min_s2 = 10
nn2 = NearestNeighbors(n_neighbors=min_s2)
nn2.fit(X_db2)
dists2, _ = nn2.kneighbors(X_db2)
dists2_sorted = np.sort(dists2[:, -1])[::-1]

plt.figure(figsize=(9, 4))
plt.plot(dists2_sorted, color='darkorange')
plt.title(f'[RONDA B] K-distancias (k={min_s2}) – datos PCA(100)')
plt.xlabel('Puntos ordenados'); plt.ylabel('Distancia')
plt.grid(True, alpha=0.3); plt.show()

eps_b = 3.5   # <-- Modificar según el codo del gráfico

dbscan_b    = DBSCAN(eps=eps_b, min_samples=min_s2, n_jobs=-1)
labels_db_b = dbscan_b.fit_predict(X_train_pca)

n_clust_b = len(set(labels_db_b)) - (1 if -1 in labels_db_b else 0)
n_ruido_b = np.sum(labels_db_b == -1)

print("=== [RONDA B] DBSCAN (PCA 100) ===")
print(f"eps={eps_b}, min_samples={min_s2}")
print(f"Clusters encontrados : {n_clust_b}")
print(f"Puntos de ruido      : {n_ruido_b} ({n_ruido_b/len(labels_db_b)*100:.1f}%)")

mask_nr_b = labels_db_b != -1
if n_clust_b >= 2 and mask_nr_b.sum() > 1:
    sil_db_b = silhouette_score(X_train_pca[mask_nr_b], labels_db_b[mask_nr_b],
                                 sample_size=3000, random_state=42)
    print(f"Silhouette Score     : {sil_db_b:.4f}")
else:
    sil_db_b = float('nan')
    print("Silhouette Score     : N/A (ajusta eps)")

ari_db_b = adjusted_rand_score(y_train, labels_db_b)
print(f"ARI                  : {ari_db_b:.4f}")


In [ ]:
# ============================================================
# CELDA 15 – RONDA B: Clustering Jerárquico sobre PCA(100)
# ============================================================
# AgglomerativeClustering Ward con n_clusters=3 sobre X_train_pca.
# Incluye dendrograma sobre muestra de 500 puntos para visualizar
# la jerarquía de fusiones y confirmar K=3 como corte óptimo.
# Métricas: Silhouette Score, ARI y matriz de confusión (verde).

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix
from scipy.cluster.hierarchy import dendrogram, linkage
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Dendrograma (muestra de 500 puntos)
np.random.seed(42)
idx_dend = np.random.choice(len(X_train_pca), size=500, replace=False)
Z = linkage(X_train_pca[idx_dend], method='ward')

plt.figure(figsize=(13, 5))
dendrogram(Z, truncate_mode='lastp', p=20, leaf_rotation=45,
           leaf_font_size=9, show_contracted=True)
plt.axhline(y=Z[-3, 2] * 0.7, color='red', linestyle='--', alpha=0.6, label='Corte K≈3')
plt.title('[RONDA B] Dendrograma – Clustering Jerárquico Ward (PCA 100)')
plt.xlabel('Cluster / Índice'); plt.ylabel('Distancia de fusión')
plt.legend(); plt.tight_layout(); plt.show()

# Clustering sobre todo el train
agg_b        = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg_b = agg_b.fit_predict(X_train_pca)

sil_agg_b = silhouette_score(X_train_pca, labels_agg_b, sample_size=3000, random_state=42)
ari_agg_b = adjusted_rand_score(y_train, labels_agg_b)

print("=== [RONDA B] Clustering Jerárquico Ward (PCA 100) ===")
print(f"Silhouette Score : {sil_agg_b:.4f}")
print(f"ARI              : {ari_agg_b:.4f}")
print(f"Distribución     : {np.bincount(labels_agg_b)}")

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_train, labels_agg_b), annot=True, fmt='d', cmap='Greens',
            xticklabels=[f'C{i}' for i in range(3)],
            yticklabels=[f'Clase {i}' for i in range(3)])
plt.title('[RONDA B] Jerárquico Ward (PCA 100): Clusters vs Clases')
plt.ylabel('Clase Real'); plt.xlabel('Cluster')
plt.show()


In [ ]:
# ============================================================
# CELDA 16: Comparativa Final No Supervisado – Ronda A vs Ronda B
# ============================================================
# Se consolidan las métricas de los 6 experimentos de clustering
# (3 algoritmos × 2 rondas) en tabla y gráfico de barras agrupadas.
# Objetivo: visualizar el impacto del PCA sobre la calidad del clustering.
#   - Silhouette Score: calidad de separación interna (sin etiquetas).
#   - ARI: alineación con las 3 clases reales de cáncer de pulmón.
# Se espera que Ronda B (PCA 100) supere a Ronda A (768D) en ambas.

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

resultados_ns = [
    {'Ronda': 'A – Crudos (768D)',  'Algoritmo': 'K-Means',    'Silhouette': sil_km_a,  'ARI': ari_km_a},
    {'Ronda': 'A – Crudos (768D)',  'Algoritmo': 'DBSCAN',     'Silhouette': sil_db_a,  'ARI': ari_db_a},
    {'Ronda': 'A – Crudos (768D)',  'Algoritmo': 'Jerárquico', 'Silhouette': sil_agg_a, 'ARI': ari_agg_a},
    {'Ronda': 'B – PCA(100)',       'Algoritmo': 'K-Means',    'Silhouette': sil_km_b,  'ARI': ari_km_b},
    {'Ronda': 'B – PCA(100)',       'Algoritmo': 'DBSCAN',     'Silhouette': sil_db_b,  'ARI': ari_db_b},
    {'Ronda': 'B – PCA(100)',       'Algoritmo': 'Jerárquico', 'Silhouette': sil_agg_b, 'ARI': ari_agg_b},
]

df_ns = pd.DataFrame(resultados_ns)
print("=== COMPARATIVA FINAL – No Supervisado ===")
print(df_ns.to_string(index=False))

# Gráfico de barras agrupadas
algoritmos = ['K-Means', 'DBSCAN', 'Jerárquico']
x = np.arange(len(algoritmos))
ancho = 0.35

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, metrica in zip(axes, ['Silhouette', 'ARI']):
    vals_a = [df_ns[(df_ns['Ronda']=='A – Crudos (768D)') & (df_ns['Algoritmo']==a)][metrica].values[0] for a in algoritmos]
    vals_b = [df_ns[(df_ns['Ronda']=='B – PCA(100)')      & (df_ns['Algoritmo']==a)][metrica].values[0] for a in algoritmos]
    ax.bar(x - ancho/2, vals_a, ancho, label='Ronda A (Crudos)',  color='#5dade2', alpha=0.85)
    ax.bar(x + ancho/2, vals_b, ancho, label='Ronda B (PCA 100)', color='#27ae60', alpha=0.85)
    ax.set_title(f'{metrica} – Ronda A vs Ronda B')
    ax.set_xticks(x); ax.set_xticklabels(algoritmos)
    ax.set_ylabel(metrica); ax.legend(); ax.grid(axis='y', alpha=0.3)

plt.suptitle('Comparativa No Supervisado: Datos Crudos vs PCA(100)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()
